In [ ]:
#| hide
from drona.core import *
from drona.rounds import *
from drona.sessions import *

# drona

> Train agents to choose and use the right tools.

Drona reads Ramabana session history, identifies poor tool routes, and creates a reviewed Urai history for the start of a new session.

## Install

```sh
pip install drona
```

## Assess Ramabana history

`assess_turn` scores observable tool actions. It does not infer quality from the assistant narrative.

In [ ]:
turn = {
    'prompt': 'Use fossick to research https://github.com/AnswerDotAI/llmdojo',
    'activity': [{'tool': 'web_search', 'ok': True, 'args': {'query': 'llmdojo'}}],
}
assessment = assess_turn(turn)
assessment

Run `drona` to assess the default Ramabana history. Pass `--session` to limit the report to one session.

## Prepare a chat

`warm_start` returns canonical Urai history. Pass it to any Urai or Rishi chat through `messages=`, or call `prepare_chat` on an empty chat.

In [ ]:
history = warm_start()
[(message['role'], message.get('name')) for message in history]

A completion receipt includes the package version and round revision. `completion_valid` rejects receipts from an older round.

## Curate a round

Ramabana already records completed turns. Capture reads that archive after the conversation ends. It does not monitor a live process.

```sh
ramabana --root /path/to/project
# finish the useful conversation, then quit
drona-capture training/github-research.ipynb --session latest
leela training
```

Open `github-research.ipynb` in Leela. Remove detours and sensitive content. Keep the route that future models should imitate. Save the notebook, then accept it with a reviewer name.

```sh
drona-accept training/github-research.ipynb --reviewer Karthik
```

Acceptance updates the notebook metadata and writes `github-research.json` as derived canonical history. An unaccepted notebook cannot start a session.

`drona-start` prints the Ramabana bootstrap and resume commands by default. `--launch` runs them.

```sh
drona-start training/github-research.ipynb --root /path/to/project
drona-start training/github-research.ipynb --root /path/to/project --launch
```

Ramabana receives the accepted round as its first bootstrap turn and saves it. Drona then resumes that session. Leela can use `compiled_history` directly when it adds prepared-history support.

## Move sessions between hosts

Aidialog notebooks are the interchange format. Ramabana, Claude, and Codex sessions can all be imported for review in Leela.

```sh
drona-import ramabana training/round.ipynb --session latest
drona-import claude training/round.ipynb --session latest --cwd /path/to/project
drona-import codex training/round.ipynb --session latest --cwd /path/to/project
```

The same reviewed notebook can become a Ramabana bootstrap, a resumable Claude session, or Codex-native Responses items.

```sh
drona-export training/round.ipynb ramabana --output training/round.txt
drona-export training/round.ipynb claude --cwd /path/to/project
drona-export training/round.ipynb codex --output training/round-items.json
```

Claude Code can resume the id printed by the Claude export. Codex export does not create a resumable rollout because llmsurgery has no public rollout writer. The exported items remain suitable for inspection, datasets, and a future Codex launcher.

## Develop

```sh
uv sync --all-extras --group dev
uv run nbdev-export
uv run nbdev-test
uv run nbdev-readme
```